In [1]:
import pandas as pd
import music21 as m21
from fractions import Fraction
from harmonic_inference.data.data_types import ChordType, PitchType, KeyMode
from harmonic_inference.utils.harmonic_utils import get_scale_degree_from_interval, get_pitch_from_string
import harmonic_inference.utils.harmonic_constants as hc
from pathlib import Path
import os

In [25]:
# dicts

chord_types = {
"dominant-seventh": ChordType.MAJ_MIN7,
"major": ChordType.MAJOR,
"major-seventh": ChordType.MAJ_MAJ7,
"diminished": ChordType.DIMINISHED,
"augmented": ChordType.AUGMENTED,
"diminished-seventh": ChordType.DIM7,
"minor": ChordType.MINOR,
"minor-seventh": ChordType.MIN_MIN7,
"half-diminished-seventh": ChordType.HALF_DIM7,
"major-sixth": ChordType.MAJOR, # not defined in ChordType class
"minor-sixth": ChordType.MINOR, # Not defined in ChordType class
"minor-ninth": ChordType.MINOR, # not defined in chordtype class
"minor-13th": ChordType.MINOR # not defined in chord class
}


CHORD_TYPES_TO_STRING_READABLE = {
    ChordType.MAJ_MIN7: "D7",
    ChordType.MAJOR: "M",
    ChordType.MAJ_MAJ7: "M7",
    ChordType.DIMINISHED: "d",
    ChordType.AUGMENTED: "a",
    ChordType.DIM7: "d7",
    ChordType.MINOR: "m",
    ChordType.MIN_MIN7: "m7",
    ChordType.HALF_DIM7: "h7",
}

In [26]:
def process_score(score, score_id, keys_df):
    """
    Process a music21 Score and extract chord information with keys from CSV.
    
    Args:
        score: music21 Score object

        score_id: score identifier string (e.g., "score_1361")

        keys_df: DataFrame with columns [score_id, key_onset_measure, key, mode]
    
    Returns:
        DataFrame with columns: on, off, key, degree, type, inv
    """
    
    # Get all harmonies in the score
    harmonies = score.flat.getElementsByClass("Harmony")
    
    if not harmonies:
        return pd.DataFrame()


    # Get actual measures from the score to map measure numbers to offsets
    all_measures = []
    for part in score.parts:
        part_measures = list(part.getElementsByClass(m21.stream.Measure))
        if part_measures:
            all_measures = part_measures
            break  # Use measures from the first part that has them

    # Load CSV keys for this score
    score_keys = keys_df[keys_df["score_id"] == score_id]
    
    key_changes = []
    
    if not score_keys.empty: # ensure there is key change data in the csv
        measure_by_number = { # this maps each measure.number from the score to its measure object, so key changes can be located by real measure number instead of list position
            measure.number: measure
            for measure in all_measures
            if measure.number is not None
        }

        
        # Build key_changes list from CSV using "actual" measure offsets
        for _, row in score_keys.iterrows():
            measure_num = int(row["key_onset_measure"])
            key_str = row["key"]
            mode = row["mode"]
            
            # Get the offset of this measure from the score
            measure = measure_by_number.get(measure_num)
            if measure is not None:
                offset = measure.offset

            
            # else:
            #     # vibecoded Fallback: try positional lookup, then proportional calculation
            #     measure_index = measure_num - 1
            #     if 0 <= measure_index < len(all_measures):
            #         offset = all_measures[measure_index].offset
            #     else:
            #         highest_harmony_offset = max(h.offset for h in harmonies) # 1. find end of piece
            #         total_measures = len(all_measures) if all_measures else score_keys["key_onset_measure"].max() # 2. Count total measures
            #         quarter_notes_per_measure = highest_harmony_offset / total_measures # 3. Calculate avg measure len
            #         offset = (measure_num - 1) * quarter_notes_per_measure # predict offset
            
            # Create music21 Key object
            major_mode = mode.lower() == "major"
            key_obj = m21.key.Key(key_str, "major" if major_mode else "minor")
            key_changes.append((offset, key_obj))
    
    # else:

    #     # vibecoded Fallback: use XML keys if no CSV keys found
    #     for k in score.flat.getElementsByClass(m21.key.Key):
    #         key_changes.append((k.offset, k))
    #     if not key_changes:
    #         for ks in score.flat.getElementsByClass(m21.key.KeySignature):
    #             key_changes.append((ks.offset, ks.asKey()))
    
    key_changes.sort(key=lambda x: x[0])
    
    # Helper function: get key object at a given offset
    def get_key_obj_at_offset(offset):  
        for ks_offset, k_obj in reversed(key_changes): # reversed() ensures we find the latest active key; iterating forward would always match the very first key change at offset 0.
            if offset >= ks_offset:  
                return k_obj
        return m21.key.Key('C')  # C as default Key object

    # Helper function: format key string
    def format_key_string(k_obj):
        tonic_name = k_obj.tonic.name
        mode = k_obj.mode
        key_str = tonic_name.upper() if mode == "major" else tonic_name.lower()  
        return key_str.replace("-", "b").replace("#", "+")  

    # Helper function: get the measure number for a harmony offset 
    # by checking if first measure in the score is shorter than a normal measure defined by the timeSignature
    pickup_measure_adjustment = 1 if all_measures and getattr(all_measures[0], "barDuration", None) and all_measures[0].duration.quarterLength < all_measures[0].barDuration.quarterLength else 0

    def get_measure_num_at_offset(offset):
        if not all_measures: # fallback
            return None
        for measure_num, measure in reversed(list(enumerate(all_measures, start=1))):
            if offset >= measure.offset:
                return max(measure_num - pickup_measure_adjustment, 0)
        return max(1 - pickup_measure_adjustment, 0) # fallback

    # Get score length (in quarter notes)
    piece_len = score.flat.highestOffset

    rows = []

    # inv = []

    # def find_chord_inversions(chordsym):
    #     chordsym.writeAsChord = True


    for harmony in harmonies: # = score.flat.getElementsByClass("Harmony")
        on = Fraction(harmony.offset)
        measure_num = get_measure_num_at_offset(on)

        # Get the correct key at this harmony's offset
        
        current_key_obj = get_key_obj_at_offset(on)
        key_str = format_key_string(current_key_obj)
        key_mode = KeyMode.MAJOR if current_key_obj.mode == "major" else KeyMode.MINOR

        # Extract chord root

        if hasattr(harmony, "root") and harmony.root() is not None:
            chord_root_string = harmony.root().name.replace("-", "b")
        else:
            chord_root_string = harmony.figure.split(":")[0].replace("-", "b")

        # Calculate roman numeral degree

        chord_root_int = get_pitch_from_string(chord_root_string.replace("+", "#"), PitchType.TPC)
        key_tonic_int = get_pitch_from_string(key_str.replace("+", "#"), PitchType.TPC)
        degree = get_scale_degree_from_interval(
            chord_root_int - key_tonic_int,
            key_mode,
            PitchType.TPC,
        )

        
        harmony_type_str = harmony.chordKind or "major" # harmony.chordKind gives us things like 'major', 'major-seventh', ... default to major?
        chord_type_enum = chord_types.get(harmony_type_str, None)

        if chord_type_enum is not None:
            chord_type = CHORD_TYPES_TO_STRING_READABLE[chord_type_enum]

        else:
            chord_type = "unknown"
        inv=0
        rows.append([on, None, key_str, degree, chord_type, inv]) # None because the shifted on will be there # / ADD "measure_num" for debug

    # Create DataFrame WITHOUT column names

    df = pd.DataFrame(rows, columns=["on", "off", "key", "degree", "type", "inv"]) # / ADD "measure_num" for debug

    # Shift the 'on' time of the next chord to become the "off" time of the current chord
    df["off"] = df["on"].shift(-1)

    # Handle the last chord's "off" time

    if not df.empty:
        df.iloc[-1, df.columns.get_loc("off")] = Fraction(piece_len)

    return df


In [27]:
# Load keys from CSV
keys_df = pd.read_csv("choro_corpus/keys_choro_pieces.csv")

# Process entire directory
input_dir = Path("choro_corpus/with_chords")
output_dir = Path("choro_model_ft/chords")

# When running with Model outputs: input dir = ("playground/original_with_removed_chords")
# output dir = Path("choro_model_ft/chords_from_model_output")
# when running with Original scores: input dir = "choro_corpus/with_chords"
# when running with Original scores: output dir = "choro_model_ft/chords"

output_dir.mkdir(exist_ok=True)

for file in input_dir.glob("*.xml"):

    try:
        score = m21.converter.parse(file)
        
        # Extract score_id from filename (everything before first dash)
        score_id = file.stem.split('-')[0]

        df = process_score(score, score_id, keys_df)

        output_file = output_dir / f"{file.stem}.csv"

        df.to_csv(
            output_file,
            index=False,
            header=False
        )
        print(f"✓ Processed {file.stem}")
    except Exception as e: 
        print(f"✗ Failed on {file.name}: {e}")


✓ Processed score_6879-Tema_de_apresentacao_da_V_Sinfonia-Beethoven
✓ Processed score_4566-Isto_nao_e_vida-Macarico
✓ Processed score_1985-Tira_Poeira-Satyro_Bilhar
✓ Processed score_1361-Medrosa-Anacleto_de_Medeiros
✓ Processed score_265-Paixao_encoberta-Mario_Alvares
✓ Processed score_2802-Belezas_do_Recife-Misael_Domingues
✓ Processed score_1425-Terna_Saudade-Anacleto_de_Medeiros
✓ Processed score_5063-Pinguim-Ernesto_Nazareth
✓ Processed score_5789-Tristeza-Eduardo_Souto
✓ Processed score_117-Bohemia_Terra-Irineu_de_Almeida
✓ Processed score_5602-Flor_do_Abacate-Alvaro_Sandim
✓ Processed score_4637-Desmantelando_Relogios-Cicero_Telles_de_Meneses
✓ Processed score_4990-Menino_de_Ouro-Ernesto_Nazareth
✓ Processed score_2622-Tu_passaste_por_este_Jardim-Alfredo_Dutra
✓ Processed score_4800-Atlantico-Ernesto_Nazareth
✓ Processed score_1235-Valsa-Joaquim_Callado
✓ Processed score_11143-Gemea-Chiquinha_Gonzaga
✓ Processed score_846-Conceicao-Joaquim_Callado
✓ Processed score_1324-Em_ti_Pe

In [72]:
# Mauro's request: change relative degree column to absolute

def process_score(score, score_id, keys_df):
    """
    Process a music21 Score and extract chord information with keys from CSV.
    
    Args:
        score: music21 Score object

        score_id: score identifier string (e.g., "score_1361")

        keys_df: DataFrame with columns [score_id, key_onset_measure, key, mode]
    
    Returns:
        DataFrame with columns: on, off, key, degree, type, inv
    """
    
    # Get all harmonies in the score
    harmonies = score.flat.getElementsByClass("Harmony")
    
    if not harmonies:
        return pd.DataFrame()

    # Get actual measures from the score to map measure numbers to offsets
    all_measures = []
    for part in score.parts:
        part_measures = list(part.getElementsByClass(m21.stream.Measure))
        if part_measures:
            all_measures = part_measures
            break  # Use measures from the first part that has them

    # Load CSV keys for this score
    score_keys = keys_df[keys_df["score_id"] == score_id]
    
    key_changes = []
    
    if not score_keys.empty: # ensure there is key change data in the csv

        measure_by_number = { # this maps each measure.number from the score to its measure object, so key changes can be located by real measure number instead of list position
            measure.number: measure
            for measure in all_measures
            if measure.number is not None
        }

        
        # Build key_changes list from CSV using "actual" measure offsets
        for _, row in score_keys.iterrows():
            measure_num = int(row["key_onset_measure"])
            key_str = row["key"]
            mode = row["mode"]
            
            # Get the offset of this measure from the score
            measure = measure_by_number.get(measure_num)
            if measure is not None:
                offset = measure.offset

            
            # else:
            #     # vibecoded Fallback: try positional lookup, then proportional calculation
            #     measure_index = measure_num - 1
            #     if 0 <= measure_index < len(all_measures):
            #         offset = all_measures[measure_index].offset
            #     else:
            #         highest_harmony_offset = max(h.offset for h in harmonies) # 1. find end of piece
            #         total_measures = len(all_measures) if all_measures else score_keys["key_onset_measure"].max() # 2. Count total measures
            #         quarter_notes_per_measure = highest_harmony_offset / total_measures # 3. Calculate avg measure len
            #         offset = (measure_num - 1) * quarter_notes_per_measure # predict offset
            
            # Create music21 Key object
            major_mode = mode.lower() == "major"
            key_obj = m21.key.Key(key_str, "major" if major_mode else "minor")
            key_changes.append((offset, key_obj))
    
    # else:

    #     # vibecoded Fallback: use XML keys if no CSV keys found
    #     for k in score.flat.getElementsByClass(m21.key.Key):
    #         key_changes.append((k.offset, k))
    #     if not key_changes:
    #         for ks in score.flat.getElementsByClass(m21.key.KeySignature):
    #             key_changes.append((ks.offset, ks.asKey()))
    
    key_changes.sort(key=lambda x: x[0])
    
    # Helper function: get key object at a given offset
    def get_key_obj_at_offset(offset):  
        for ks_offset, k_obj in reversed(key_changes): # reversed() ensures we find the latest active key; iterating forward would always match the very first key change at offset 0.
            if offset >= ks_offset:  
                return k_obj
        return m21.key.Key('C')  # C as default Key object

    # Helper function: format key string
    def format_key_string(k_obj):
        tonic_name = k_obj.tonic.name
        mode = k_obj.mode
        key_str = tonic_name.upper() if mode == "major" else tonic_name.lower()  
        return key_str.replace("-", "b").replace("#", "+")  

    # Helper function: get the measure number for a harmony offset 
    # by checking if first measure in the score is shorter than a normal measure defined by the timeSignature
    pickup_measure_adjustment = 1 if all_measures and getattr(all_measures[0], "barDuration", None) and all_measures[0].duration.quarterLength < all_measures[0].barDuration.quarterLength else 0

    def get_measure_num_at_offset(offset):
        if not all_measures: # fallback
            return None
        for measure_num, measure in reversed(list(enumerate(all_measures, start=1))):
            if offset >= measure.offset:
                return max(measure_num - pickup_measure_adjustment, 0)
        return max(1 - pickup_measure_adjustment, 0) # fallback

    # Get score length (in quarter notes)
    piece_len = score.flat.highestOffset

    rows = []

    # inv = []

    # def find_chord_inversions(chordsym):
    #     chordsym.writeAsChord = True


    for harmony in harmonies: # = score.flat.getElementsByClass("Harmony")

        on = Fraction(harmony.offset)
        measure_num = get_measure_num_at_offset(on)

        # Get the correct key at this harmony's offset
        
        current_key_obj = get_key_obj_at_offset(on)
        key_str = format_key_string(current_key_obj)
        key_mode = KeyMode.MAJOR if current_key_obj.mode == "major" else KeyMode.MINOR

        # Extract chord root

        if hasattr(harmony, "root") and harmony.root() is not None:
            chord_root_string = harmony.root().name.replace("-", "b")
        else:
            chord_root_string = harmony.figure.split(":")[0].replace("-", "b")

        # Calculate roman numeral degree

        # chord_root_int = get_pitch_from_string(chord_root_string.replace("+", "#"), PitchType.TPC)
        # key_tonic_int = get_pitch_from_string(key_str.replace("+", "#"), PitchType.TPC)
        # degree = get_scale_degree_from_interval(
        #     chord_root_int - key_tonic_int,
        #     key_mode,
        #     PitchType.TPC,
        # )
        
        if hasattr(harmony, "root") and harmony.root() is not None:
            degree = harmony.root().name
            degree = chord_root_string
        
        harmony_type_str = harmony.chordKind or "major" # harmony.chordKind gives us things like 'major', 'major-seventh', ... default to major?
        chord_type_enum = chord_types.get(harmony_type_str, None)

        if chord_type_enum is not None:
            chord_type = CHORD_TYPES_TO_STRING_READABLE[chord_type_enum]

        else:
            chord_type = "unknown"
        inv=0
        rows.append([on, None, key_str, degree, chord_type, inv]) # None because the shifted on will be there # / ADD "measure_num" for debug

    # Create DataFrame WITHOUT column names

    df = pd.DataFrame(rows, columns=["on", "off", "key", "degree", "type", "inv"]) # / ADD "measure_num" for debug

    # Shift the 'on' time of the next chord to become the "off" time of the current chord
    df["off"] = df["on"].shift(-1)

    # Handle the last chord's "off" time

    if not df.empty:
        df.iloc[-1, df.columns.get_loc("off")] = Fraction(piece_len)

    return df


In [ ]:
# Load keys and test on a single score
keys_df = pd.read_csv("choro_corpus/keys_choro_pieces.csv")
test_score = m21.converter.parse("choro_corpus/with_chords/score_3791-Plangente-Chiquinha_Gonzaga.xml")
process_score(test_score, "score_3791", keys_df)


# Everything below are older versions / tests!